<a href="https://colab.research.google.com/github/bdaniel01/AAI2026-BDD/blob/main/Prompt_Engineer/Ex3_Self_Reflection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 3: Self-Reflection Prompt for Improving Output

**Tools Used:** Google Colab, Python, Gemini API

**Goal:** Generate an initial summary, critique it against specific requirements, and revise it based on that critique.

**Self-Reflection Flow:** Initial Summary → Self-Critique → Revised Summary


In [1]:
# ==============================================================================
# Exercise 3: Self-Reflection Prompt for Improving Output
# Environment: Google Colab
# Library: google-genai
# ==============================================================================

!pip install -q google-genai

import time
from google import genai
from google.genai import errors
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# Gemini 2.5 Flash worked successfully in Exercises 1 and 2
PRIMARY_MODEL = "gemini-2.5-flash"


def safe_generate_content(prompt):
    for attempt in range(5):
        try:
            return client.models.generate_content(
                model=PRIMARY_MODEL,
                contents=prompt
            )

        except errors.ServerError:
            time.sleep(2 ** attempt)

        except errors.APIError as e:
            if e.code == 429:
                wait_time = 10 + (attempt * 5)
                time.sleep(wait_time)
            else:
                raise RuntimeError(f"API Error ({e.code}): {e.message}")

        except Exception as e:
            if getattr(e, "code", None) == 429 or "429" in str(e):
                wait_time = 10 + (attempt * 5)
                time.sleep(wait_time)
            else:
                raise RuntimeError(f"Unexpected error: {e}")

    raise RuntimeError(
        f"Failed to generate content with model {PRIMARY_MODEL} after retries."
    )


# ==============================================================================
# SOURCE ARTICLE
# ==============================================================================

source_article = """Many retailers are adopting artificial intelligence to improve inventory
forecasting, an area that has traditionally depended on historical sales,
seasonal patterns, and manager experience. Modern forecasting systems can
combine point-of-sale data with promotions, local events, weather patterns,
online search activity, and supplier information. Machine learning models can
use these signals to estimate future demand at the product and store level.

The business value comes from reducing two expensive problems: overstock and
stockouts. Overstock ties up cash and may force retailers to discount products
that do not sell quickly. Stockouts create the opposite problem because
customers cannot purchase an item when they want it, which can reduce sales and
customer satisfaction. More accurate forecasts can help retailers place orders
closer to expected demand and make better decisions about how inventory should
be distributed across locations.

However, AI forecasting is not automatically accurate. The quality of the
output depends heavily on the quality and relevance of the data used to train
and update the model. Sudden events can also make historical patterns less
useful. A system trained during normal conditions may struggle when consumer
behavior changes because of a major storm, supply disruption, economic shift,
or unexpected social trend.

Organizations also need employees who can interpret the model's
recommendations. Forecasting systems may identify patterns that are difficult
to explain, so managers still need business knowledge to decide whether a
prediction makes sense. Human review is especially important when a decision
could lead to a large purchase, major markdowns, or changes across many stores.

For executives, the main challenge is not simply choosing an AI tool. Companies
must decide how the system fits into existing purchasing and supply-chain
processes, how performance will be measured, who is responsible for reviewing
recommendations, and how employees will respond when the model is wrong. AI can
support faster and more data-driven inventory decisions, but its value depends
on combining technology with reliable data, clear processes, and human
judgment."""

print("=== SOURCE TEXT ===")
print(source_article.strip())
print("=" * 60)


# ------------------------------------------------------------------------------
# STEP 1: Initial Generation
# ------------------------------------------------------------------------------

initial_prompt = f"""
Summarize the following business article about AI inventory forecasting.

Article:
{source_article}
"""

initial_response = safe_generate_content(initial_prompt)
before_summary = initial_response.text

print("\n--- BEFORE SUMMARY (INITIAL GENERATION) ---")
print(before_summary.strip())

time.sleep(5)


# ------------------------------------------------------------------------------
# STEP 2: Self-Reflection & Critique Step
# ------------------------------------------------------------------------------

critique_prompt = f"""
You are an executive communications editor.

Perform a strict critique of the Original Summary by comparing it directly
against the Source Article.

Source Article:
{source_article}

Original Summary:
{before_summary}

Rubric Criteria for Critique:

1. Accuracy & Focus:
   - Does the summary accurately preserve the most important business and
     technical ideas from the article?
   - Does it avoid adding unsupported information?
   - Does it explain the business value of reducing overstock and stockouts?
   - Does it preserve the importance of data quality and human judgment?

2. Clarity:
   - Is the wording clear, concise, and easy to understand?
   - Does it avoid unnecessary jargon?

3. Target Audience & Tone:
   - Is the summary appropriate for a C-suite or executive audience?
   - Does it focus on business impact and decision-making?

4. Length & Formatting:
   - Is the summary under 150 words total?
   - Is it structured as EXACTLY 3 bullet points?

For each criterion, clearly state what PASSED and what FAILED.
Then briefly explain what should be improved in the revised version.

Do not rewrite the summary yet.
"""

critique_response = safe_generate_content(critique_prompt)
self_critique = critique_response.text

print("\n--- SELF-CRITIQUE OUTPUT ---")
print(self_critique.strip())

time.sleep(5)


# ------------------------------------------------------------------------------
# STEP 3: Revision Step Based on Critique
# ------------------------------------------------------------------------------

revision_prompt = f"""
You are an expert business editor.

Revise the original summary based on the self-critique below.

Source Article:
{source_article}

Original Summary:
{before_summary}

Self-Critique:
{self_critique}

Mandatory Output Constraints:
- MUST be formatted as EXACTLY 3 bullet points.
- MUST be under 150 words in total.
- Tone must be concise, professional, and appropriate for executives.
- Preserve the most important business and technical ideas from the source.
- Include the business value of reducing overstock and stockouts.
- Include the importance of data quality and human judgment.
- Do not add unsupported information.

Output ONLY the final revised summary.
"""

revision_response = safe_generate_content(revision_prompt)
after_summary = revision_response.text

print("\n--- AFTER SUMMARY (REVISED GENERATION) ---")
print(after_summary.strip())


# ------------------------------------------------------------------------------
# SIMPLE BEFORE / AFTER VERIFICATION
# ------------------------------------------------------------------------------

before_word_count = len(before_summary.split())
after_word_count = len(after_summary.split())

before_bullets = sum(
    1 for line in before_summary.splitlines()
    if line.strip().startswith(("-", "*", "•"))
)

after_bullets = sum(
    1 for line in after_summary.splitlines()
    if line.strip().startswith(("-", "*", "•"))
)

print("\n--- FINAL CHECK ---")
print(f"Before word count: {before_word_count}")
print(f"After word count: {after_word_count}")
print(f"Before bullet count: {before_bullets}")
print(f"After bullet count: {after_bullets}")
print("Under 150 words:", "PASS" if after_word_count < 150 else "FAIL")
print("Exactly 3 bullet points:", "PASS" if after_bullets == 3 else "FAIL")


=== SOURCE TEXT ===
Many retailers are adopting artificial intelligence to improve inventory
forecasting, an area that has traditionally depended on historical sales,
seasonal patterns, and manager experience. Modern forecasting systems can
combine point-of-sale data with promotions, local events, weather patterns,
online search activity, and supplier information. Machine learning models can
use these signals to estimate future demand at the product and store level.

The business value comes from reducing two expensive problems: overstock and
stockouts. Overstock ties up cash and may force retailers to discount products
that do not sell quickly. Stockouts create the opposite problem because
customers cannot purchase an item when they want it, which can reduce sales and
customer satisfaction. More accurate forecasts can help retailers place orders
closer to expected demand and make better decisions about how inventory should
be distributed across locations.

However, AI forecasting is n